# Transfer Learning

Previously we built our own CNN from scratch, which after 10 epochs did OK:

```output
Baseline CNN from scratch
Accuracy:          72.7%
Balanced accuracy: 34.3%
Macro F1:          36.9%
```


In this notebook lets use ResNet-18 pretrained on ImageNet. It's relatively small, well understood, and appropriate for your 4 GB RTX 3050.

It doesn't already "know melanoma." Instead, earlier layers have learned broadly useful visual representations such as edges, textures, shapes, and progressively more complex patterns.

Note: We use ResNet instead of VGG because it is quite a bit smaller:

```output
ResNet-18       ~11.7 million parameters
VGG-16         ~138 million parameters
```

VGG has many more parameters in the final classification layer.  Hence, ResNet is better suited to my meager RTX GPU with only 4 GB of VRAM.

### Data Pipeline


STart by re-using the same data ingest pipeline up through the Dataset and Dataloader definitions.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

print("PyTorch:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [5]:
PROJECT_ROOT = Path.cwd().parent

METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "metadata_splits.csv"
)

df = pd.read_csv(METADATA_PATH)

# Image paths are stored relative to the project root
df["image_path"] = df["image_path"].map(lambda p: str(PROJECT_ROOT / p))

print(df.shape)
print(df["split"].value_counts())

(10015, 9)
split
train    7002
val      1532
test     1481
Name: count, dtype: int64


In [6]:
#Define the classes
CLASS_NAMES = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

class_to_idx = {
    name: idx for idx, name in enumerate(CLASS_NAMES)
}

idx_to_class = {
    idx: name for name, idx in class_to_idx.items()
}

NUM_CLASSES = len(CLASS_NAMES)

print(class_to_idx)

{'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}


In [7]:
#Recreate data input pipeline
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [8]:
#Dataset
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[row["dx"]]

        return image, label

In [9]:
#Dataloaders
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

train_dataset = HAM10000Dataset(train_df, train_transform)
val_dataset = HAM10000Dataset(val_df, eval_transform)
test_dataset = HAM10000Dataset(test_df, eval_transform)

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

Now instead of building our own CNN from scratch, we start with the pre-trained ResNet one.

In [ ]:
#Load prae-trained ResNet
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT

model = resnet18(weights=weights)

print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/msmith/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 65.4MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [2]:
#Of course we need to replace the last layer, which is designed for 1000 ImageNet classes
print(model.fc)

Linear(in_features=512, out_features=1000, bias=True)


In [10]:
#Reaplce the last layer 
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    NUM_CLASSES
)

model = model.to(device)

print(model.fc)

Linear(in_features=512, out_features=7, bias=True)


In [11]:
#Freeze all of the parameters
for param in model.parameters():
    param.requires_grad = False

#But then unfreeze the new classifier (fc) layer
for param in model.fc.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters:     11,180,103
Trainable parameters: 3,591


In [ ]:
#Build loss and optimizer
criterion = nn.CrossEntropyLoss()

#Optimizer ONLY grabs the parameters of the new classifier (fc) layer
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=1e-3
)

In [13]:
#Verify with one batch
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input:", images.shape)
print("Output:", outputs.shape)
print("Labels:", labels.shape)

Input: torch.Size([32, 3, 224, 224])
Output: torch.Size([32, 7])
Labels: torch.Size([32])


In [14]:
#Inspect loss
loss = criterion(outputs, labels)

print(f"Initial loss: {loss.item():.4f}")

Initial loss: 1.9692


In [16]:
#Train one epoch
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [17]:
#And for validation
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [18]:
#Train for 5 epochs
NUM_EPOCHS = 5

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(NUM_EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(
        f"Epoch {epoch + 1:2d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.3f}"
    )

Epoch  1/5 | Train Loss: 0.9297 | Train Acc: 0.691 | Val Loss: 0.8873 | Val Acc: 0.677
Epoch  2/5 | Train Loss: 0.7608 | Train Acc: 0.732 | Val Loss: 0.7698 | Val Acc: 0.725
Epoch  3/5 | Train Loss: 0.7106 | Train Acc: 0.739 | Val Loss: 0.7246 | Val Acc: 0.743
Epoch  4/5 | Train Loss: 0.6883 | Train Acc: 0.750 | Val Loss: 0.7144 | Val Acc: 0.752
Epoch  5/5 | Train Loss: 0.6737 | Train Acc: 0.755 | Val Loss: 0.6964 | Val Acc: 0.757


In [19]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().tolist()
        )

        all_labels.extend(
            labels.tolist()
        )

In [20]:
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score
)

print(
    classification_report(
        all_labels,
        all_predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0,
    )
)

balanced_acc = balanced_accuracy_score(
    all_labels,
    all_predictions
)

print(
    f"Balanced Accuracy: {balanced_acc:.3f}"
)

              precision    recall  f1-score   support

       akiec      0.667     0.314     0.427        51
         bcc      0.421     0.584     0.489        77
         bkl      0.561     0.408     0.472       157
          df      1.000     0.053     0.100        19
         mel      0.478     0.324     0.386       170
          nv      0.847     0.932     0.888      1034
        vasc      0.424     0.583     0.491        24

    accuracy                          0.757      1532
   macro avg      0.628     0.457     0.465      1532
weighted avg      0.745     0.757     0.738      1532

Balanced Accuracy: 0.457


Balanced accuracy of 45% better than stage 1 baseline CNN of 34%.

### Fine Tuning

So far we have kept the ResNet model as is, only replacing the last classification layer.  This is transfer learning.

Now lets do some fine tuning by allowing some of the convolutional layer parameters to change, even if only some of the later layers and even if only changing them slowly with lower learning rate.  Changing just the last conv layer will allow it to adapt it to learn visual features that are well-suited to classifying lesions.

In [21]:
#Unfreeze layer 4
for param in model.layer4.parameters():
    param.requires_grad = True

In [22]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(
    f"Percent trainable: "
    f"{100 * trainable_params / total_params:.1f}%"
)

Total parameters:     11,180,103
Trainable parameters: 8,397,319
Percent trainable: 75.1%


In [25]:
#Update optimizer with all trainable parameters
optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-4
)
#We've also lowered learning rate from 1e-3 to 1e-4
#This allows for slower "fine tuning"

#Also confirm still have correct loss function
criterion = nn.CrossEntropyLoss()

In [26]:
#Train another 5 epochs
NUM_EPOCHS = 5

ft_train_losses = []
ft_val_losses = []
ft_train_accuracies = []
ft_val_accuracies = []

for epoch in range(NUM_EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    ft_train_losses.append(train_loss)
    ft_val_losses.append(val_loss)
    ft_train_accuracies.append(train_acc)
    ft_val_accuracies.append(val_acc)

    print(
        f"Epoch {epoch + 1:2d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.3f}"
    )

Epoch  1/5 | Train Loss: 0.6027 | Train Acc: 0.784 | Val Loss: 0.6016 | Val Acc: 0.783
Epoch  2/5 | Train Loss: 0.4593 | Train Acc: 0.827 | Val Loss: 0.5819 | Val Acc: 0.792
Epoch  3/5 | Train Loss: 0.4063 | Train Acc: 0.848 | Val Loss: 0.6011 | Val Acc: 0.790
Epoch  4/5 | Train Loss: 0.3458 | Train Acc: 0.873 | Val Loss: 0.6868 | Val Acc: 0.772
Epoch  5/5 | Train Loss: 0.3079 | Train Acc: 0.886 | Val Loss: 0.6442 | Val Acc: 0.785


In [28]:
#Updated classification report after fine tuning
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().tolist()
        )

        all_labels.extend(
            labels.tolist()
        )

print(
    classification_report(
        all_labels,
        all_predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0,
    )
)

balanced_acc = balanced_accuracy_score(
    all_labels,
    all_predictions
)
print(f"Balanced Accuracy: {balanced_acc:.3f}")

              precision    recall  f1-score   support

       akiec      0.541     0.392     0.455        51
         bcc      0.600     0.701     0.647        77
         bkl      0.524     0.701     0.599       157
          df      0.375     0.474     0.419        19
         mel      0.469     0.406     0.435       170
          nv      0.926     0.893     0.909      1034
        vasc      0.630     0.708     0.667        24

    accuracy                          0.785      1532
   macro avg      0.581     0.611     0.590      1532
weighted avg      0.793     0.785     0.786      1532

Balanced Accuracy: 0.611


Recap so far:

```output
Random initialization
Baseline CNN
Balanced accuracy = 34.3%
        │
        │ introduce ImageNet features
        ▼
Frozen ResNet-18
Balanced accuracy = 45.7%
        │
        │ adapt high-level features
        │ to dermatoscopic images
        ▼
Fine-tuned ResNet-18
Balanced accuracy = 61.1%
```

In [ ]:
#Save the fine-tuned model
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_to_idx": class_to_idx,
        "architecture": "resnet18",
        "training_strategy": "ImageNet pretrained; layer4 + fc fine-tuned",
        "val_accuracy": 0.785,
        "val_balanced_accuracy": 0.611,
        "val_macro_f1": 0.590,
    },
    MODEL_DIR / "resnet18_finetuned.pt"
)
#About 40 MB

In [30]:
#Save off classification report as well
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(
    all_labels,
    all_predictions,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).transpose()

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

report_df.to_csv(
    RESULTS_DIR / "resnet18_finetuned_val_report.csv"
)

report_df

,precision,recall,f1-score,support
akiec,0.540541,0.392157,0.454545,51.000000
bcc,0.600000,0.701299,0.646707,77.000000
bkl,0.523810,0.700637,0.599455,157.000000
df,0.375000,0.473684,0.418605,19.000000
mel,0.469388,0.405882,0.435331,170.000000
nv,0.925777,0.892650,0.908912,1034.000000
vasc,0.629630,0.708333,0.666667,24.000000
accuracy,0.784595,0.784595,0.784595,0.784595
macro avg,0.580592,0.610663,0.590032,1532.000000
weighted avg,0.793271,0.784595,0.786467,1532.000000


### Update Loss Function to account for class imbalance

Ordinary CrossEntropyLoss gives every **image** equal weight, which then gives more weight to the more prevelant classes.

In [31]:
#See weight classes
import numpy as np
import torch

class_counts = (
    train_df["dx"]
    .value_counts()
    .reindex(CLASS_NAMES)
)

print(class_counts)

dx
akiec     230
bcc       366
bkl       774
df         76
mel       778
nv       4679
vasc       99
Name: count, dtype: int64


In [32]:
num_samples = len(train_df)
num_classes = len(CLASS_NAMES)

class_weights = (
    num_samples /
    (num_classes * class_counts.values)
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

for class_name, count, weight in zip(
    CLASS_NAMES,
    class_counts,
    class_weights.cpu().numpy()
):
    print(
        f"{class_name:5s} | "
        f"count: {count:4d} | "
        f"weight: {weight:.3f}"
    )

akiec | count:  230 | weight: 4.349
bcc   | count:  366 | weight: 2.733
bkl   | count:  774 | weight: 1.292
df    | count:   76 | weight: 13.162
mel   | count:  778 | weight: 1.286
nv    | count: 4679 | weight: 0.214
vasc  | count:   99 | weight: 10.104


So we assign much more weight to less common classes, effectively giving each class the same overall weight (Senate vs House of Representatives).

In practice, a loss of 1 for df will add 13 to the loss, whereas same loss of 1 for popular nv image will contribute only 0.214 to the overall loss.

In [33]:
#Update loss function with these weights
weighted_criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

In [ ]:
#Start from same starting point:
# Pretrained ResNet-18 model
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT

weighted_model = resnet18(weights=weights)

#OVERWRITE the final classification layer
weighted_model.fc = nn.Linear(
    weighted_model.fc.in_features,
    NUM_CLASSES
)

weighted_model = weighted_model.to(device)

In [35]:
#Freeze parameters
for param in weighted_model.parameters():
    param.requires_grad = False

#Unfreeze final classication layer
for param in weighted_model.fc.parameters():
    param.requires_grad = True

In [36]:
#Update opimizer, only grab the fc layer parameters
weighted_optimizer = torch.optim.Adam(
    weighted_model.fc.parameters(),
    lr=1e-3
)

Interesting point here on the trainable parameters.  Here we have two redundant ways to ensure that only the fc parameters are updated:
1. Set `requires_grad = False` for everything except final `fc` layer
2. Optimizer only grabs `weighted_model.fc.parameters()`

It turns out we don't strictly need to do that, as having either one of these on their own would do the same task of only updated the fc parameters.  For example, if we still did step 1, but had optimizer grab all parameters with `weighted_model.parameters()`, then that would still work as pytorch would prevent the optimizer from changing the earlier parameters (or from calculating gradients since that's the definition of requires_grad = False).

Of course, good idea to be explicit so that the intentions are clear, and also is good to ensure robust and consistent behavior.

In [37]:
NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):

    train_loss, train_acc = train_one_epoch(
        weighted_model,
        train_loader,
        weighted_criterion,
        weighted_optimizer,
        device
    )

    val_loss, val_acc = evaluate(
        weighted_model,
        val_loader,
        weighted_criterion,
        device
    )

    print(
        f"Epoch {epoch + 1:2d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.3f}"
    )

Epoch  1/5 | Train Loss: 1.6009 | Train Acc: 0.505 | Val Loss: 1.4545 | Val Acc: 0.502
Epoch  2/5 | Train Loss: 1.2841 | Train Acc: 0.607 | Val Loss: 1.2664 | Val Acc: 0.548
Epoch  3/5 | Train Loss: 1.1946 | Train Acc: 0.617 | Val Loss: 1.0689 | Val Acc: 0.627
Epoch  4/5 | Train Loss: 1.1325 | Train Acc: 0.636 | Val Loss: 1.0539 | Val Acc: 0.625
Epoch  5/5 | Train Loss: 1.0964 | Train Acc: 0.637 | Val Loss: 1.5220 | Val Acc: 0.475


In [38]:
#Now fine tune by making layer4 trainable as well
for param in weighted_model.layer4.parameters():
    param.requires_grad = True

#Update optimizer to include layer4 parameters, again with lower learning rate
weighted_optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        weighted_model.parameters()
    ),
    lr=1e-4
)

In [39]:
#Train another 5 epochs to do fine tuning
# Using weighted criterion
NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):

    train_loss, train_acc = train_one_epoch(
        weighted_model,
        train_loader,
        weighted_criterion,
        weighted_optimizer,
        device
    )

    val_loss, val_acc = evaluate(
        weighted_model,
        val_loader,
        weighted_criterion,
        device
    )

    print(
        f"Epoch {epoch + 1:2d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.3f}"
    )

Epoch  1/5 | Train Loss: 1.0030 | Train Acc: 0.646 | Val Loss: 1.0147 | Val Acc: 0.640
Epoch  2/5 | Train Loss: 0.7013 | Train Acc: 0.725 | Val Loss: 0.8404 | Val Acc: 0.684
Epoch  3/5 | Train Loss: 0.5984 | Train Acc: 0.749 | Val Loss: 0.9438 | Val Acc: 0.659
Epoch  4/5 | Train Loss: 0.5196 | Train Acc: 0.771 | Val Loss: 0.8395 | Val Acc: 0.694
Epoch  5/5 | Train Loss: 0.4414 | Train Acc: 0.787 | Val Loss: 0.8156 | Val Acc: 0.723


In [40]:
#Updated classification report after fine tuning
weighted_model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = weighted_model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().tolist()
        )

        all_labels.extend(
            labels.tolist()
        )

print(
    classification_report(
        all_labels,
        all_predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0,
    )
)

balanced_acc = balanced_accuracy_score(
    all_labels,
    all_predictions
)
print(f"Balanced Accuracy: {balanced_acc:.3f}")

              precision    recall  f1-score   support

       akiec      0.491     0.510     0.500        51
         bcc      0.479     0.870     0.618        77
         bkl      0.515     0.662     0.579       157
          df      0.391     0.474     0.429        19
         mel      0.354     0.535     0.426       170
          nv      0.951     0.768     0.850      1034
        vasc      0.773     0.708     0.739        24

    accuracy                          0.723      1532
   macro avg      0.565     0.647     0.591      1532
weighted avg      0.791     0.723     0.745      1532

Balanced Accuracy: 0.647


Note that overall accuracy has gone *down*, but balanced accuracy across classes has gone up.

| Metric | Baseline CNN | ResNet fine-tuned | ResNet + weighted loss |
|---|---:|---:|---:|
| Accuracy | 72.7% | **78.5%** | 72.3% |
| Balanced accuracy | 34.3% | 61.1% | **64.7%** |
| Macro F1 | 36.9% | 59.0% | **59.1%** |

In [41]:
#Save class-weighted model
torch.save(
    {
        "model_state_dict": weighted_model.state_dict(),
        "class_to_idx": class_to_idx,
        "architecture": "resnet18",
        "training_strategy": (
            "ImageNet pretrained; layer4 + fc fine-tuned; "
            "class-weighted cross-entropy"
        ),
        "val_accuracy": 0.723,
        "val_balanced_accuracy": 0.647,
        "val_macro_f1": 0.591,
    },
    MODEL_DIR / "resnet18_finetuned_weighted.pt"
)

### Fine tune on entire model

So far we have done transfer learning (overwriting the final classification layer to predict our 7 classes rather than the 1000 ImageNet classes), and have also done fine tuning with just the last conv layer.

Lets experiment with fine tuning and changing the entire ResNet model.  Note that it's not clear going in if this will help or hurt.  Maybe the fine tuning will adapt the earlier layers to find features that are helpful for this problem, or maybe it will lead to overfitting and the image and feature recognition capabilities from ImageNet are already sufficient.

Start by loading the UNWEIGHTED fine tuned model.  This is the original ResNet model where we added the new classification layer, trained that for 5 epochs, then unfroze the last conv layer and fine tuned for an additional 5 epochs, all while using regular cross entropy loss that assigns equal weight to every image.

In [42]:
#Reload the class-weighted model
from torchvision.models import resnet18

#Re-build the STRUCUTURE of the full fine-tuned model
full_ft_model = resnet18(weights=None)

full_ft_model.fc = nn.Linear(
    full_ft_model.fc.in_features,
    NUM_CLASSES
)

#Now load the model and the state_dict containing the value of the weights
checkpoint = torch.load(
    MODEL_DIR / "resnet18_finetuned.pt",
    map_location=device
)

full_ft_model.load_state_dict(
    checkpoint["model_state_dict"]
)

full_ft_model = full_ft_model.to(device)

Note on saving and loading pytorch models. 

In this approach, we save only the state_dict containing the parameter values, which means when we want to re-load it we need to first build the empty architecture (with random weights), and then load in the learned parameter values from the state_dict with

```python
torch.save(
    model.state_dict(),
    "model.pt"
)

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 7)

model.load_state_dict(
    torch.load("model.pt")
)
```

We could save the ENTIRE model with 

```python
torch.save(
    model,
    "model_full.pt"
)
```

and then reload it later with

```python
model = torch.load(
    "model_full.pt",
    weights_only=False
)
```


While saving and loading entire model is convenient, it is also more brittle as it can break if using different environment or package versions.  Hence, the state_dict approach tends to be preferable.

Note that in this case, we save the state_dict along with additional information and metadata:

```python
torch.save(
    {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "class_to_idx": class_to_idx,
        "architecture": "resnet18",
    },
    "checkpoint.pt"
)
```

This "checkpoint" approach is more reproducible and portfolio-friendly than relying on a pickled full-model object, so makes sense to use here.  It is also better suited to deploying models as we'll see later.

OK, we have `full_ft_model`, lets train it.

In [43]:
#Unfreeze EVERYTHING
for param in full_ft_model.parameters():
    param.requires_grad = True

total_params = sum(
    p.numel() for p in full_ft_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in full_ft_model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters:     11,180,103
Trainable parameters: 11,180,103


In [44]:
#Return to regular cross entropy, not class-weighted
criterion = nn.CrossEntropyLoss()

#Use smaller learning rate in optimizer
full_ft_optimizer = torch.optim.Adam(
    full_ft_model.parameters(),
    lr=1e-5
)

In [45]:
#Train for 5 more epochs
NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):

    train_loss, train_acc = train_one_epoch(
        full_ft_model,
        train_loader,
        criterion,
        full_ft_optimizer,
        device
    )

    val_loss, val_acc = evaluate(
        full_ft_model,
        val_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch + 1:2d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.3f}"
    )

Epoch  1/5 | Train Loss: 0.2327 | Train Acc: 0.915 | Val Loss: 0.6285 | Val Acc: 0.792
Epoch  2/5 | Train Loss: 0.2141 | Train Acc: 0.920 | Val Loss: 0.6136 | Val Acc: 0.798
Epoch  3/5 | Train Loss: 0.1903 | Train Acc: 0.931 | Val Loss: 0.6164 | Val Acc: 0.798
Epoch  4/5 | Train Loss: 0.1820 | Train Acc: 0.935 | Val Loss: 0.6142 | Val Acc: 0.802
Epoch  5/5 | Train Loss: 0.1688 | Train Acc: 0.941 | Val Loss: 0.6322 | Val Acc: 0.796


In [46]:
#Evaluate model on validation data and print results
full_ft_model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = full_ft_model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().tolist()
        )
        all_labels.extend(labels.tolist())



In [47]:
print(
    classification_report(
        all_labels,
        all_predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0,
    )
)

balanced_acc = balanced_accuracy_score(
    all_labels,
    all_predictions
)

print(f"Balanced Accuracy: {balanced_acc:.3f}")

              precision    recall  f1-score   support

       akiec      0.556     0.588     0.571        51
         bcc      0.583     0.727     0.647        77
         bkl      0.654     0.637     0.645       157
          df      0.625     0.263     0.370        19
         mel      0.449     0.412     0.429       170
          nv      0.907     0.911     0.909      1034
        vasc      0.615     0.667     0.640        24

    accuracy                          0.796      1532
   macro avg      0.627     0.601     0.602      1532
weighted avg      0.794     0.796     0.793      1532

Balanced Accuracy: 0.601


We're seeing some overfitting, as training acc improves while validaiton plateaus, even getting *worse* after epoch 2.  We may end up taking model after best epoch, not necessarily the last epoch (known as *checkpointing after validation performance*).  For that to work, we actually have to *save* a copy of the model after each epoch.  As is, there's no way to revert to what the model looked like after epoch 2.  Can put a pin in this concept for later.

In [48]:
#Save model
torch.save(
    {
        "model_state_dict": full_ft_model.state_dict(),
        "class_to_idx": class_to_idx,
        "architecture": "resnet18",
        "training_strategy": "full network fine-tuning",
        "val_accuracy": 0.796,
        "val_balanced_accuracy": 0.601,
        "val_macro_f1": 0.602,
    },
    MODEL_DIR / "resnet18_full_finetuned.pt"
)